# OpenMontage Kaggle Batch Asset Generator

This kernel runs on Kaggle's GPU infrastructure and generates all image/video
assets for a single OpenMontage project in one session. It is submitted by the
Codespace tool layer and polled until completion.

## Behavior contract

- Reads job tasks directly from the embedded `JOB_TASKS` dict (no external files).
- Detects GPU compute capability and selects the appropriate model branch.
- Hard-fails on any unrecoverable error — never silently substitutes placeholders.

In [ ]:
import os, sys, json, time, shutil, base64, hashlib
from pathlib import Path
from datetime import datetime, timezone
import torch

# TEST FLAG - TEMPORARY: force T4-only for this diagnostic run
# REVERT AFTER: set to False and restore original three-branch logic
TEST_FORCE_T4 = True

# =============================================================================
# HF_TOKEN FALLBACK CHAIN
# Priority: Kaggle Secrets > os.environ > empty string
# Kaggle kernels do NOT inherit the Codespace .env, so the user MUST set
# HF_TOKEN as an actual Kaggle Secret (Settings -> Add-ons -> Secrets).
# =============================================================================
HF_TOKEN = os.environ.get('HF_TOKEN', '')

# Try Kaggle secrets next - this is the primary expected path for Kaggle kernels
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_from_secrets = user_secrets.get_secret('HF_TOKEN')
    if hf_from_secrets:
        HF_TOKEN = hf_from_secrets
        print('HF_TOKEN from Kaggle Secrets successfully loaded')
    else:
        print('HF_TOKEN secret not found for key "HF_TOKEN"')
except Exception as e:
    print('HF_TOKEN secret access error:', repr(e))  # Debug the exact exception

if not HF_TOKEN:
    print('WARNING: HF_TOKEN is empty. HF Hub downloads may hit rate limits.')

# Set token for huggingface_hub
os.environ['HF_TOKEN'] = HF_TOKEN

In [ ]:
# =============================================================================
# GPU DETECTION
# Determines which model branch to use based on compute capability.
# TEST_FORCE_T4 enforces T4 requirement for SANA-Sprint testing.
# =============================================================================
print('Detecting GPU...')
GPU_BRANCH = 'cpu'
GPU_NAME = 'cpu'
DEVICE = 'cpu'

if torch.cuda.is_available():
    cc = torch.cuda.get_device_properties(0).major
    GPU_NAME = torch.cuda.get_device_name(0)
    
    if TEST_FORCE_T4:
        # TEMPORARY: Force T4 requirement for this diagnostic run
        if cc < 7:
            print(f'ERROR: T4 enforcement triggered - GPU {GPU_NAME} (CC={cc}) incompatible with SANA-Sprint')
            print('T4 or better required (compute capability >= 7)')
            sys.exit(1)
        GPU_BRANCH = 't4_or_better'
        DEVICE = 'cuda'
        print(f'T4 GPU enforced: {GPU_NAME} (CC={cc})')
    elif cc >= 7:
        GPU_BRANCH = 't4_or_better'
        DEVICE = 'cuda'
        print(f'GPU detected: {GPU_NAME} (compute capability {cc})')
        print('Branch: SANA-Sprint (steps=4, guidance_scale=4.5, 1024x576)')
    elif cc == 6:
        GPU_BRANCH = 'p100'
        DEVICE = 'cuda'
        print(f'GPU detected: {GPU_NAME} (compute capability {cc})')
        print('Branch: FLUX.1 Schnell FP8 (P100 incompatible with SANA)')
    else:
        GPU_BRANCH = 'old_gpu'
        DEVICE = 'cpu'
        print(f'GPU detected but CC={cc} not supported, falling back to CPU')
else:
    print('No GPU detected. Branch: FLUX.1 Schnell FP8 CPU-only mode')

print(f'Active branch: {GPU_BRANCH}')
print(f'Device: {DEVICE}')
print(f'HF_TOKEN present: {bool(HF_TOKEN)}')

In [ ]:
# =============================================================================
# SINGLETON MODEL LOADING
# Load pipelines once and reuse across all assets in the batch
# _SANA_PIPE uses SANA-Sprint (1.6B, 1024px, 4 steps)
# _FLUX_PIPE uses FLUX.1 Schnell (fallback for P100/CPU)
# =============================================================================
_SANA_PIPE = None
_FLUX_PIPE = None
def _get_sana_pipe():
    """Load SANA-Sprint pipeline once per kernel session (singleton pattern)."""
    global _SANA_PIPE
    if _SANA_PIPE is None:
        from diffusers import SanaSprintPipeline
        import torch
        _SANA_PIPE = SanaSprintPipeline.from_pretrained(
            'Efficient-Large-Model/Sana_Sprint_1.6B_1024px_diffusers',
            torch_dtype=torch.bfloat16 if DEVICE == 'cuda' else torch.float32,
            device_map='cuda' if DEVICE == 'cuda' else None,
        )
        if DEVICE == 'cuda':
            _SANA_PIPE.vae.to(torch.float32)
    return _SANA_PIPE

def _get_flux_pipe():
    """Load FLUX pipeline once per kernel session (singleton pattern)."""
    global _FLUX_PIPE
    if _FLUX_PIPE is None:
        from diffusers import FluxPipeline
        import torch
        _FLUX_PIPE = FluxPipeline.from_pretrained(
            'black-forest-labs/FLUX.1-schnell',
            torch_dtype=torch.float32 if DEVICE == 'cuda' else torch.float32,
            device_map='cuda' if DEVICE == 'cuda' else None,
        )
        if DEVICE == 'cpu':
            _FLUX_PIPE.enable_model_cpu_offload()
        # Warm-up and offload text encoder to CPU to save ~1.5GB VRAM
        with torch.no_grad():
            _ = _FLUX_PIPE.text_encoder.encode('test')
            torch.cuda.empty_cache()
            import gc
            gc.collect()
    return _FLUX_PIPE


In [ ]:
JOB_TASKS = {
    "project_id": "jeju-cold-case",
    "channel": "crime-ledger",
    "assets": [
        {
            "type": "image",
            "id": "scene_01",
            "prompt": "rain-soaked Jeju street at 3 AM, taxi pulling away, empty road, monochrome noir, film grain",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_02",
            "prompt": "taxi dispatch phone, hand pressing numbers, dim screen glow, forensic document, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_03",
            "prompt": "taxi interior rear seat, empty passenger view, rain streaks, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_04",
            "prompt": "evidence bag with wallet and phone, chain of custody, forensic table, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_05",
            "prompt": "empty drainage ditch, bare earth, overcast sky, forensic marker, clinical restraint",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_06",
            "prompt": "medical examiner report, neck diagram, ligature mark annotation, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_07",
            "prompt": "autopsy timeline graph, conflicting date arrows, coroner note, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_08",
            "prompt": "taxi driver lineup, five thousand files, suspect photo with red string, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_09",
            "prompt": "CCTV still frame, white taxi at intersection, timestamp, red circle annotation",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_10",
            "prompt": "microscope fiber evidence, jacket texture comparison, forensic document, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_11",
            "prompt": "fiber batch stamp, mass production chart, evidence inconsistency, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_12",
            "prompt": "phone call log with gaps, deletion marker, suspicious but inconclusive, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_13",
            "prompt": "illegal search warrant, suppressed evidence stamp, redacted report, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_14",
            "prompt": "evidence table with empty slots, circumstantial cards, clinical restraint, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_15",
            "prompt": "empty courtroom dock, gavel, ten-year case file, film grain, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_16",
            "prompt": "not guilty verdict headline, newspaper clipping, empty dock, film grain, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_17",
            "prompt": "appeal brief stack, cumulative evidence arrows, clinical monochrome, typewriter labels",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_18",
            "prompt": "appellate courtroom, second not guilty stamp, film grain, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_19",
            "prompt": "appellate verdict document, reasonable doubt highlight, monochrome documentary",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_20",
            "prompt": "Supreme Court building, legal brief, standard of proof question, dark monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_21",
            "prompt": "Supreme Court confirmation stamp, mass-produced fibers document, case unresolved, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_22",
            "prompt": "statute expiration clock, 2015 abolition key unused, time-bound justice, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_23",
            "prompt": "relocation map, assumed identity document, nine-year gap, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_24",
            "prompt": "Park silhouette, microphone, destroyed life quote, film grain, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_25",
            "prompt": "family portrait, empty justice scales, Supreme Court document, dark moody, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_26",
            "prompt": "TAWAN Act document, statute clock reset, hope and family portrait, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_27",
            "prompt": "cold case team meeting, evidence board, re-examination markers, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_28",
            "prompt": "conflicting timeline chart, autopsy versus police arrows, contradiction, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_29",
            "prompt": "decomposition experiment, pig and dog specimens, forensic scientists, clinical restraint",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_30",
            "prompt": "decomposition timeline, animal to human arrow, uncertainty margin, scientific diagram",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_31",
            "prompt": "arrest warrant, police car lights, nine-year gap closed, monochrome noir",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_32",
            "prompt": "defense table, Park silhouette, denial, empty prosecution table, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_33",
            "prompt": "target fiber report, mass production stamp, defense objection, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_34",
            "prompt": "eighteen Sonata grid, CCTV highlight, probability not identity, diagram style",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_35",
            "prompt": "deleted call records, routine management argument, suspicious, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_36",
            "prompt": "empty evidence table, missing weapon, no DNA match, uncertain CCTV, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_37",
            "prompt": "verdict document, reasonable doubt highlighted, inference not proof, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_38",
            "prompt": "appeal brief, cumulative evidence chart, insufficient verdict, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_39",
            "prompt": "second appeal verdict, proof standard bar, evidence falling short, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_40",
            "prompt": "Supreme Court brief, cumulative impact diagram, reversal request, dark monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_41",
            "prompt": "Supreme Court dismissal, fiber exclusion logic, case closed legally, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_42",
            "prompt": "double jeopardy wall, final acquittal seal, closed courthouse, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_43",
            "prompt": "unanswered question mark, empty taxi, foggy road, unresolved file, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_44",
            "prompt": "second taxi driver silhouette, witness report, deprioritized lead, dark mystery, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_45",
            "prompt": "Supreme Court excerpt, third taxi highlighted, reasonable doubt, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_46",
            "prompt": "timeline branching diagram, two taxi routes, murder location uncertainty, scientific diagram",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_47",
            "prompt": "anonymous witness file, untested vehicle icon, unverified route, dark mystery, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_48",
            "prompt": "confirmation bias diagram, spotlight on Park, second taxi in shadow, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_49",
            "prompt": "five thousand canvass map, empty second taxi slot, theatrical spotlight, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_50",
            "prompt": "pig dog decomposition chart, species difference warning, extrapolation arrow, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_51",
            "prompt": "target fiber lab, innovation versus validation scale, microscope, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_52",
            "prompt": "eighteen Sonata grid, CCTV highlight, probability not identity, diagram style",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_53",
            "prompt": "deleted call log, innocent versus guilty Venn, reasonable doubt scale, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_54",
            "prompt": "relocation path, stalled investigation sign, nine-year gap, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_55",
            "prompt": "cold case team, re-examination markers, new evidence not proof, documentary",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_56",
            "prompt": "experiment result chart, needle moving, court record original estimate, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_57",
            "prompt": "Supreme Court flowchart, fiber exclusion logic, doubt shield, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_58",
            "prompt": "double jeopardy wall, retrial blocked sign, new evidence barrier, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_59",
            "prompt": "young woman portrait, childcare teacher, certain fact, taxi home route, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_60",
            "prompt": "investigation resources, man-years clock, forensic tools, unsatisfying outcome, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_61",
            "prompt": "solitary figure, no microphone, aftermath landscape, dark moody, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_62",
            "prompt": "family grief, February first calendar, wrong answer stamp, ongoing sorrow, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_63",
            "prompt": "film poster overlay, Jeju landscape, fiction-reality mirror, dark ambient, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_64",
            "prompt": "empty evidence table, five missing slots, inference argument, clinical restraint, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_65",
            "prompt": "five thousand canvass map, empty second taxi slot, map gaps, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_66",
            "prompt": "anonymous witness file, unaccounted vehicle, untraced route, dark mystery, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_67",
            "prompt": "relocation path, assumed identity document, nine-year gap, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_68",
            "prompt": "cold case team formation, new tests, evidence re-examination, documentary",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_69",
            "prompt": "experiment results, needle moving, error margin, clinical monochrome, scientific",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_70",
            "prompt": "target fiber lab, innovation badge, validation question mark, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_71",
            "prompt": "Park weeping, courtroom, destroyed life quote, film grain, monochrome noir",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_72",
            "prompt": "family portrait, Supreme Court document, empty justice scales, dark moody, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_73",
            "prompt": "officially closed stamp, expiration document, unresolved file, dark ambient, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_74",
            "prompt": "film poster reference, Jeju landscape, haunting question mark, dark ambient, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_75",
            "prompt": "anonymous witness protection, sealed file, unmapped route, dark mystery, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_76",
            "prompt": "cold case file, forensic flaw icon, evidence limit boundary, police question mark, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_77",
            "prompt": "first-of-its-kind badge, precedent law book, limitation warning, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_78",
            "prompt": "fiber connection web, suggestive link, not sufficient red X, clinical monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_79",
            "prompt": "eighteen Sonata grid, CCTV highlight, probability not identity, diagram style",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_80",
            "prompt": "deleted records, innocent versus guilty diagram, reasonable doubt scale, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_81",
            "prompt": "court reasoning flowchart, circumstantial pyramid, alternative explanation breaking point, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_82",
            "prompt": "Park statement quote, destroyed life visual, irreversible damage, film grain, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_83",
            "prompt": "family in gallery, empty conviction seat, system functioning but failing, dark moody, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_84",
            "prompt": "Jeju map, trust fracture lines, community fear, dark ambient, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_85",
            "prompt": "empty evidence table, five missing slots, inference argument, clinical restraint, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_86",
            "prompt": "five thousand canvass grid, empty second taxi slot, map gaps, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_87",
            "prompt": "anonymous witness waveform, sealed statement, unaccounted vehicle, dark mystery, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_88",
            "prompt": "young woman portrait, childcare teacher, certain fact, taxi home route, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_89",
            "prompt": "investigation resources, man-years clock, forensic tools, unsatisfying outcome, monochrome",
            "width": 1024,
            "height": 576
        },
        {
            "type": "image",
            "id": "scene_90",
            "prompt": "mostly empty evidence board, second taxi shadow, unknown perpetrator, unresolved file, distant siren, monochrome film grain",
            "width": 1024,
            "height": 576
        }
    ]
}
OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Job tasks loaded: {len(JOB_TASKS.get("assets", []))} assets to generate')


## Hard-fail safeguard

Any generation failure stops the job immediately and returns a non-zero exit
code. There is NO silent substitution of placeholder images. The caller should
inspect the generated manifest and abort on any missing asset.

In [ ]:
def write_manifest(results: list, error: str | None = None) -> dict:
    """Write the asset manifest to disk.
    
    Returns the manifest dict. If error is set, the manifest includes
    failure markers for every asset.
    """
    manifest = {
        'project_id': JOB_TASKS['project_id'],
        'channel': JOB_TASKS['channel'],
        'gpu_branch': GPU_BRANCH,
        'gpu_name': GPU_NAME,
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'error': error,
        'assets': results,
    }
    manifest_path = OUTPUT_DIR / 'manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return manifest


def fail_job(message: str) -> None:
    """Hard-fail the job with a clear error message."""
    print(f'[FATAL] {message}')
    write_manifest([], error=message)
    sys.exit(1)

In [ ]:
# =============================================================================
# IMAGE GENERATION
# Uses SANA-Sprint on T4+, FLUX.1 Schnell on P100 or CPU.
# =============================================================================

from PIL import Image
import io

def channel_to_negative_prompt(channel: str) -> str:
    """Map channel styles to anti-anachronism negative prompts."""
    negatives = {
        'dark-annals': 'modern clothing, electric lighting, smartphones, digital screens, anachronistic architecture, contemporary technology',
        'crime-ledger': 'contemporary fashion, neon signs, mixed-era tech, photorealistic',
        'mind-tactics': 'futuristic interfaces, holograms, cyberpunk accessories, neon',
    }
    return negatives.get(channel, '')

def augment_prompt_for_sana(base_prompt: str, channel: str) -> str:
    """Fold anti-anachronism negative concepts into the positive prompt.
    
    SANA-Sprint does not accept negative_prompt, so we append negating
    language to the positive prompt as a workaround.
    """
    neg = channel_to_negative_prompt(channel)
    if not neg:
        return base_prompt
    clauses = [f'NO {item.strip()}' for item in neg.split(',')]
    return f'{base_prompt}. {", ".join(clauses)}' 

def generate_image(task: dict) -> dict:
    """Generate a single image asset with VRAM/duration tracking."""
    import time
    asset_id = task['id']
    channel = JOB_TASKS.get('channel', 'unknown')
    negative_prompt = channel_to_negative_prompt(channel)
    
    # Load pipeline once per session (singleton)
    pipe = _get_sana_pipe() if GPU_BRANCH == 't4_or_better' else _get_flux_pipe()
    
    # Start timer AFTER pipe retrieval (excludes one-time model load)
    start = time.time()
    
    # Reset memory stats before each generation
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    with torch.no_grad():
        if GPU_BRANCH == 't4_or_better':
            sana_prompt = augment_prompt_for_sana(task['prompt'], channel)
            image = pipe(
                prompt=sana_prompt,
                height=task.get('height', 576),
                width=task.get('width', 1024),
                num_inference_steps=2,
                guidance_scale=4.5,
            ).images[0]
        else:
            image = pipe(
                prompt=task['prompt'],
                height=task.get('height', 576),
                width=task.get('width', 1024),
                negative_prompt=negative_prompt,
                num_inference_steps=4,
                guidance_scale=0.0,
            ).images[0]
    
    # Calculate metrics
    duration_s = time.time() - start
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        mem_peak = torch.cuda.max_memory_allocated()
        mem_current = torch.cuda.memory_allocated()
        mem_used = mem_peak if mem_peak > 0 else mem_current
    else:
        mem_used = 0
    
    # Cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    import gc
    gc.collect()
    
    out_path = OUTPUT_DIR / f'{asset_id}.png'
    image.save(out_path)
    
    print(f'  VRAM peak: {mem_used/1e9:.2f} GB, duration: {duration_s:.2f}s')
    
    return {
        'id': asset_id,
        'type': 'image',
        'path': str(out_path),
        'success': True,
        'memory_peak_bytes': mem_used,
        'duration_s': round(duration_s, 2),
    }


In [ ]:
# =============================================================================
# MAIN GENERATION LOOP
# =============================================================================

def main():
    if not JOB_TASKS.get('assets'):
        print('No assets to generate.')
        write_manifest([], error='empty_job')
        sys.exit(0)
    
    results = []
    assets = JOB_TASKS['assets']
    total = len(assets)
    
    for idx, task in enumerate(assets, 1):
        asset_id = task.get('id', f'asset_{idx}')
        asset_type = task.get('type', 'image')
        print(f'[{idx}/{total}] Generating {asset_type}: {asset_id}')
        
        try:
            if asset_type == 'image':
                result = generate_image(task)
            else:
                fail_job(f'Unsupported asset type: {asset_type}')
            
            if not result.get('success'):
                fail_job(f'Generation failed for {asset_id}: {result}')
            
            # Verify output file actually exists
            out = Path(result['path'])
            if not out.exists() or out.stat().st_size < 1024:
                fail_job(f'Generated file missing or empty for {asset_id}: {out}')
            
            results.append(result)
            print(f'  OK -> {out} ({out.stat().st_size} bytes)')
            
        except Exception as exc:
            fail_job(f'Exception generating {asset_id}: {exc}')
    
    manifest = write_manifest(results)
    print(f'\nComplete. Generated {len(results)}/{total} assets.')
    print(f'Manifest: {OUTPUT_DIR / "manifest.json"}')


if __name__ == '__main__':
    main()